In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import joblib

import sys
import os
sys.path.append(os.path.abspath('..'))

from BettingStrategy.ModelStrategy.LogisticRegression.get_train_test import TrainTestBuilder
from betting_strategy import betting_pipeline, seperate_bets_dfs
import shap

c:\Users\jcmar\my_files\SportsBetting\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
date = '2026-03-14'
# upcoming_fp = fr'C:\Users\jcmar\my_files\SportsBetting\Data\upcoming_events\event_dfs\upcoming_odds_stats_{date}.csv'

upcoming_fp = r'C:\Users\jcmar\my_files\SportsBetting\Data\upcoming_events\event_dfs\DEBUG_GROUP2026-03-14.csv'
upcoming_df = pd.read_csv(upcoming_fp)  

upcoming_df['math_red'] = upcoming_df['math_red'].astype('category')
upcoming_df['math_blue'] = upcoming_df['math_blue'].astype('category')
upcoming_df['elo_pred'] = upcoming_df['elo_pred'].astype('category')


In [32]:
upcoming_df.shape

(14, 298)

In [3]:
model_open = sm.load("C:\\Users\\jcmar\\my_files\\SportsBetting\\data\\saved_models\\logit_model_open.pkl")
model_close1 = sm.load("C:\\Users\\jcmar\\my_files\\SportsBetting\\data\\saved_models\\logit_model_close1.pkl")
model_close2 = sm.load("C:\\Users\\jcmar\\my_files\\SportsBetting\\data\\saved_models\\logit_model_close2.pkl")

scaler_open = joblib.load("C:\\Users\\jcmar\\my_files\\SportsBetting\\data\\saved_models\\scaler_open.pkl")
scaler_close1 = joblib.load("C:\\Users\\jcmar\\my_files\\SportsBetting\\data\\saved_models\\scaler_close1.pkl")
scaler_close2 = joblib.load("C:\\Users\\jcmar\\my_files\\SportsBetting\\data\\saved_models\\scaler_close2.pkl")

open_feats = [
                  'proba_fair_open_diff', 'reach_diff', 
                  
                  'sub_att_pm_red', 'sub_att_pm_blue',
                  'ratio_control_diff',

                  'td_landed_pm_diff',  
                  'ratio_td_diff', 
                  'adjusted_td_red', 'adjusted_td_blue',

                  'sig_str_absorbed_total_diff', 
                  'sig_str_accuracy_pct_diff',
                  'sig_str_defense_pct_diff',
                  'adjusted_sig_str_blue', 'adjusted_sig_str_red', 
                  
                  'win_pct_red', 'win_pct_blue',
                  'win_streak_diff', 'lose_streak_diff',
                  'elo_red', 'elo_blue', 'elo_pred', 'age_red', 'age_blue'
                  ]


close1_feats = [
                  'proba_fair_close1_diff', 'proba_fair_open_diff', 'reach_diff', 
                  'td_landed_total_diff', 'ratio_control_diff', 'sig_str_absorbed_total_diff', 'sig_str_landed_pm_diff', 
                  'sig_str_defense_pct_diff', 'td_attempted_pm_diff',
                  'lose_streak_diff', 'win_pct_diff', 'kd_pm_diff', 'win_streak_diff', 'ko_wins_diff', 'ko_losses_diff', 
                  'elo_red', 'elo_blue', 'age_red', 'age_blue'
                  ]

close2_feats = [
                  'proba_fair_close2_diff', 'proba_fair_open_diff', 'reach_diff', 
                  'td_landed_total_diff', 'ratio_control_diff', 'sig_str_absorbed_total_diff', 'sig_str_landed_pm_diff', 
                  'sig_str_defense_pct_diff', 'td_attempted_pm_diff',
                  'win_pct_diff', 
                  'elo_red', 'elo_blue', 'age_red', 'age_blue'
                  ]

feats_list = [open_feats, close1_feats, close2_feats]
model_list = [model_open, model_close1, model_close2]
scaler_list = [scaler_open, scaler_close1, scaler_close2]

type_list = ['open', 'close1', 'close2']
fair_odds_list = [['dec_fair_open_blue', 'dec_fair_open_red'], ['dec_fair_close1_blue', 'dec_fair_close1_red'], ['dec_fair_close2_blue', 'dec_fair_close2_red']]
real_odds_list = [['dec_open_blue', 'dec_open_red'], ['dec_close1_blue', 'dec_close1_red'], ['dec_close2_blue', 'dec_close2_red'] ]

df_bets_all, df_parlay_all = betting_pipeline(upcoming_df, 
                                            feats_list=feats_list, model_list=model_list, scaler_list=scaler_list, type_list=type_list,
                                            fair_odds_list=fair_odds_list, real_odds_list=real_odds_list, 
                                            bankroll=500, max_drawdown=0.25, N=2000)



here
here
here


In [6]:
df_bets_all[['open_red', 'open_blue', 'close1_red', 'close1_blue', 'close2_red', 'close2_blue']] = upcoming_df[['open_red', 'open_blue', 'close1_red', 'close1_blue', 'close2_red', 'close2_blue']]

df_bets_arr, df_parlay_arr = seperate_bets_dfs(df_bets_all, df_parlay_all, type_list)
df_bets_arr[0].head(20)

,fighter_red,fighter_blue,pred_name_open,pred_winner_open,choice_proba_open,open_red,open_blue,fstar_open,stake_open,ev_open,edge_open
0,brad tavares,eryk anders,brad tavares,1,0.635487,-185.0,160.0,0.000000,0.000000,0.001707,-0.013636
1,amanda lemos,gillian robertson,gillian robertson,0,0.749350,210.0,-250.0,0.102246,51.123229,0.071373,0.035064
2,ion cutelaba,oumar sy,oumar sy,0,0.637636,170.0,-200.0,0.000000,0.000000,-0.020262,-0.029031
3,andre fili,jose delgado,jose delgado,0,0.737829,220.0,-260.0,0.056179,28.089690,0.041569,0.015606
4,marwan rahiki,harry hardwick,harry hardwick,0,NaN,-200.0,170.0,0.000000,0.000000,NaN,NaN
5,vitor petrino,steven asplund,vitor petrino,1,0.715177,-240.0,205.0,0.031601,15.800716,0.033289,0.009295
6,charles johnson,bruno silva,charles johnson,1,0.685055,-150.0,130.0,0.081569,40.784334,0.173074,0.085055
7,chris curtis,myktybek orolbai,myktybek orolbai,0,0.793733,210.0,-250.0,0.109248,54.623922,0.134829,0.079448
8,bolaji oki,manoel sousa,manoel sousa,0,NaN,170.0,-200.0,0.000000,0.000000,NaN,NaN
9,luan lacerda,hecher sosa,hecher sosa,0,NaN,170.0,-200.0,0.000000,0.000000,NaN,NaN


In [13]:
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587
EMAIL_FROM = "jcmarkufc@gmail.com"  # Gmail sender
EMAIL_PASSWORD = 'drby lvag rcjn wwsc'  # App password from GitHub secret

EMAIL_TO = "jcmarkowicz@outlook.com"

msg = MIMEText('test')
msg['Subject'] = f"Upcoming Odds Stats for {'test'}"
msg['From'] = EMAIL_FROM
msg['To'] = EMAIL_TO

# Send email
with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
    server.starttls()
    server.login(EMAIL_FROM, EMAIL_PASSWORD)
    server.send_message(msg)

In [8]:
df_parlay_arr[0].head()

,choice_fighter_name_open,parlay_fstar_open,parlay_odds_open,stake_open,parlay_ev_open,parlay_prob_open
0,charles johnson,0.175549,2.027734,87.774423,0.355966,0.447849
1,piera rodriguez,0.175549,2.027734,87.774423,0.355966,0.447849


In [10]:
df_bets_arr[0][df_bets_arr[0]['fstar_open'] != 0]

,fighter_red,fighter_blue,pred_name_open,pred_winner_open,choice_proba_open,open_red,open_blue,fstar_open,stake_open,ev_open,edge_open
2,israel adesanya,joe pyfer,joe pyfer,0,0.667242,-110.0,-110.0,0.050686,25.342764,0.334484,0.143433


In [27]:
df_bets_arr[1].head(20)

,fighter_red,fighter_blue,pred_name_close1,pred_winner_close1,choice_proba_close1,close1_red,close1_blue,fstar_close1,stake_close1,ev_close1,edge_close1
0,natalia silva,rose namajunas,natalia silva,1,0.807789,-500.0,310.0,0.000000,0.000000,-0.000232,-0.025545
1,kayla harrison,amanda nunes,kayla harrison,1,0.709203,-245.0,130.0,0.000000,0.000000,0.091481,-0.000942
2,sean omalley,song yadong,sean omalley,1,0.661578,-225.0,163.0,0.000000,0.000000,-0.000636,-0.030730
3,waldo cortes acosta,derrick lewis,waldo cortes acosta,1,0.798426,-333.0,225.0,0.075261,37.630292,0.078303,0.029373
4,ateba gautier,andrey pulyaev,ateba gautier,1,0.832048,-1200.0,500.0,0.000000,0.000000,-0.077144,-0.091029
5,arnold allen,jean silva,jean silva,0,0.645086,163.0,-280.0,0.000000,0.000000,-0.065748,-0.091757
6,umar nurmagomedov,deiveson figueiredo,umar nurmagomedov,1,0.920489,-2500.0,700.0,0.000000,0.000000,-0.027304,-0.041049
7,michael johnson,alexander hernandez,alexander hernandez,0,0.689310,145.0,-215.0,0.021327,10.663345,0.070454,0.006771
8,nikita krylov,modestas bukauskas,modestas bukauskas,0,0.571107,130.0,-195.0,0.000000,0.000000,-0.077394,-0.089910
9,alex perez,charles johnson,charles johnson,0,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN


In [122]:
df_bets_arr[2].head(20)

,fighter_red,fighter_blue,pred_name_close2,pred_winner_close2,choice_proba_close2,close2_red,close2_blue,fstar_close2,stake_close2,ev_close2,edge_close2
0,natalia silva,rose namajunas,natalia silva,1,0.787143,-430.0,350.0,0.000000,0.000000,-0.015695,-0.024178
1,kayla harrison,amanda nunes,kayla harrison,1,0.674967,-155.0,190.0,0.051849,25.924444,0.073083,0.067124
2,sean omalley,song yadong,sean omalley,1,0.670930,-200.0,175.0,0.012789,6.394392,0.026321,0.004263
3,waldo cortes acosta,derrick lewis,waldo cortes acosta,1,0.777148,-300.0,270.0,0.070952,35.475969,0.047044,0.027148
4,ateba gautier,andrey pulyaev,ateba gautier,1,0.830059,-800.0,600.0,0.000000,0.000000,-0.056423,-0.058830
5,arnold allen,jean silva,jean silva,0,0.672887,235.0,-225.0,0.000000,0.000000,-0.033337,-0.019420
6,umar nurmagomedov,deiveson figueiredo,umar nurmagomedov,1,0.915955,-1408.0,950.0,0.000000,0.000000,-0.011502,-0.017732
7,michael johnson,alexander hernandez,alexander hernandez,0,0.683407,165.0,-170.0,0.053980,26.989824,0.090770,0.053777
8,nikita krylov,modestas bukauskas,modestas bukauskas,0,0.534449,155.0,-150.0,0.000000,0.000000,-0.114512,-0.065551
9,alex perez,charles johnson,charles johnson,0,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN
